# 🎤 Fine-tune wav2vec2 for Vietnamese Children's English Pronunciation

**Goal:** Train an ASR model that accurately evaluates how well children
pronounce English words. The model converts audio → transcription, then we
score against the expected word.

**Runtime:** GPU (T4, ~2h) · **VRAM:** ~8GB · **No GPU cost** via Google Colab

---

## Step 1: Setup

Mount Google Drive and install dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

# Install dependencies
!pip install transformers datasets huggingface_hub librosa jiwer -q
print('Dependencies installed!')

## Step 2: Authenticate with HuggingFace

Get your HF token from https://huggingface.co/settings/tokens
Create a write token so you can push the model.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # Will prompt for token

## Step 3: Load Dataset

Upload `datasets/pronunciation_dataset/` from your repo to Google Drive,
or copy via gdown from a shared Drive folder.

**Expected structure:**
```
pronunciation_dataset/
├── train.json   ← list of {file, text, word, ...}
├── val.json
├── test.json
└── audio/       ← .webm audio files
```

In [ ]:
import os
import json
from pathlib import Path
from datasets import Dataset, DatasetDict, Audio

# Point to your dataset folder
DATASET_DIR = Path('/content/drive/MyDrive/edu-platform/datasets/pronunciation_dataset')

assert DATASET_DIR.exists(), f'Dataset not found at {DATASET_DIR}'
print(f'Dataset: {DATASET_DIR}')
print(f'Train: {len(json.loads((DATASET_DIR / "train.json").read_text()))} samples')
print(f'Val:   {len(json.loads((DATASET_DIR / "val.json").read_text()))} samples')
print(f'Test:  {len(json.loads((DATASET_DIR / "test.json").read_text()))} samples')

## Step 4: Config

Adjust these based on your dataset size and GPU memory.

In [ ]:
MODEL_NAME = "facebook/wav2vec2-base"          # Base model to fine-tune
LANGUAGE = "en"                                # English ASR
SAMPLE_RATE = 16_000                           # wav2vec2 requirement

# Training
EPOCHS = 20
BATCH_SIZE = 8                                 # Adjust for GPU memory
LEARNING_RATE = 1e-4
WARMUP_STEPS = 500

# Hub
HF_USERNAME = "your-hf-username"             # CHANGE THIS
MODEL_ID = f"{HF_USERNAME}/vi-child-en-pronunciation"
PUSH_TO_HUB = True                             # Set False to skip upload

## Step 5: Build HuggingFace Dataset

In [ ]:
def load_split(split_name: str) -> Dataset:
    manifest = json.loads((DATASET_DIR / f"{split_name}.json").read_text())
    audio_paths = []
    texts = []
    for item in manifest:
        path = str(DATASET_DIR / item["file"])
        if Path(path).exists():
            audio_paths.append(path)
            texts.append(item["text"])
    ds = Dataset.from_dict({"audio": audio_paths, "text": texts})
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    return ds

raw_datasets = DatasetDict({
    "train": load_split("train"),
    "validation": load_split("val"),
    "test": load_split("test"),
})

print(raw_datasets)
print("\nSample:", raw_datasets["train"][0])

## Step 6: Load Processor & Model

In [ ]:
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor,
    Wav2Vec2FeatureExtractor, Wav2Vec2CTCTokenizer,
    TrainingArguments, Trainer, DataCollatorCTCWithPadding,
)
import torch

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)
model.freeze_feature_encoder()
print(f"Model: {MODEL_NAME}")
print(f"GPU: {torch.cuda.is_available()}, {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 7: Preprocess Dataset

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    # audio is a dict with "array" and "sampling_rate"
    input_values = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_values[0]
    with processor.as_target_processor():
        labels = processor(batch["text"]).input_ids
    return {"input_values": input_values, "labels": labels}

print("Preprocessing (may take a few minutes)...")
encoded_datasets = raw_datasets.map(
    prepare_dataset,
    remove_columns=raw_datasets["train"].column_names,
    num_proc=4,
)
print("Done!")
print(encoded_datasets)

## Step 8: Training Arguments + Data Collator

In [ ]:
import numpy as np
from datasets import load_metric

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)
wer_metric = load_metric("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids)
    label_str = processor.tokenizer.batch_decode(pred.label_ids)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

OUTPUT_DIR = f"./wav2vec2-finetuned-vi-child-en"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    group_by_length=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=EPOCHS,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    save_total_limit=2,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=encoded_datasets["train"],
    eval_dataset=encoded_datasets["validation"],
    tokenizer=processor,
)
print("Trainer ready!")

## Step 9: Train!

**Expected time:** ~2-3 hours on T4 GPU.
**Expected WER:** 15-30% (children's speech is challenging; WER < 25% is great)

In [ ]:
trainer.train()

## Step 10: Evaluate on Test Set

In [ ]:
test_results = trainer.evaluate(encoded_datasets["test"])
print(f"Test WER: {test_results['eval_wer']:.4f} ({test_results['eval_wer']*100:.1f}%)")
print(f"Test Loss: {test_results['eval_loss']:.4f}")

## Step 11: Push to HuggingFace Hub

After pushing, update `HF_PRONUNCIATION_MODEL` in your backend `.env` to use the new model.

In [ ]:
if PUSH_TO_HUB:
    trainer.push_to_hub(MODEL_ID, commit_message="Fine-tuned on Vietnamese children pronunciation data")
    processor.push_to_hub(MODEL_ID)
    print(f"\n✅ Model pushed to: https://huggingface.co/{MODEL_ID}")
else:
    trainer.save_model("./local-model")
    processor.save_pretrained("./local-model")
    print("\nModel saved to ./local-model")

# Update your .env:
print(f"\n📝 Add to backend/.env:")
print(f"HF_PRONUNCIATION_MODEL={MODEL_ID}")
print(f"HF_TOKEN=your-hf-token")

---

## Quick Test: Transcribe an Audio Sample

Test your fine-tuned model on a sample.

In [ ]:
import librosa

# Load a test sample
sample = raw_datasets["test"][0]
audio = sample["audio"]["array"]

# Transcribe
input_values = processor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_values
with torch.no_grad():
    logits = model(input_values).logits
pred_ids = torch.argmax(logits, dim=-1)
transcription = processor.decode(pred_ids[0])

print(f"Expected: {sample['text']}")
print(f"Predicted: {transcription}")